# 字典的现代用法

## 字典推导式

字典推导式可从任何可迭代的 Key:Value 对构建 `dict` 实例。

In [1]:
dial_codes = [
    (880, "India"),
    (86, "China"),
    (86, "Taiwan"),
    (852, "Hong Kong"),
    (84, "Vietnam"),
    (82, "Japan"),
    (81, "Japan"),
    (80, "United Kingdom"),
    (8, "United States"),
    (7, "Russia"),
    (6, "Belarus"),
    (5, "Ukraine"),
]

country_dial = {country: code for code, country in dial_codes}
country_dial

{'India': 880,
 'China': 86,
 'Taiwan': 86,
 'Hong Kong': 852,
 'Vietnam': 84,
 'Japan': 81,
 'United Kingdom': 80,
 'United States': 8,
 'Russia': 7,
 'Belarus': 6,
 'Ukraine': 5}

In [4]:
{code : country.upper() for country,code in sorted(country_dial.items()) if code < 70}

{6: 'BELARUS', 7: 'RUSSIA', 5: 'UKRAINE', 8: 'UNITED STATES'}

In [5]:
# 测试学习

print(f"测试items:{country_dial.items()}")

print(f"排序items:{sorted(country_dial.items())}")  # 是按照key排序的

测试items:dict_items([('India', 880), ('China', 86), ('Taiwan', 86), ('Hong Kong', 852), ('Vietnam', 84), ('Japan', 81), ('United Kingdom', 80), ('United States', 8), ('Russia', 7), ('Belarus', 6), ('Ukraine', 5)])
排序items:[('Belarus', 6), ('China', 86), ('Hong Kong', 852), ('India', 880), ('Japan', 81), ('Russia', 7), ('Taiwan', 86), ('Ukraine', 5), ('United Kingdom', 80), ('United States', 8), ('Vietnam', 84)]


## 映射解包（`**`）

### 什么是解包

**解包**的意思是：把容器1里的元素拆开，摊平到当前语境中

例如序列解包
```python
print(*[1,2,3]) # 等价于print(1,2,3)
```
`*` 拆的是 序列 （元素按位置摊开）；而`**` 拆的是 映射 （键值对摊开）

### 函数调用可对多个参数应用 **

In [9]:
def dump(**kwargs):
    return kwargs

dump(**{'x':1},y=2,**{'z':3})

{'x': 1, 'y': 2, 'z': 3}

dump(**{'x': 1}, y=2, **{'z': 3}) 在执行时完全等价于：

```python
dump(x=1, y=2, z=3)
```

函数定义中的 **kwargs（参数打包）：形参前面的 ** 会将传入的所有命名参数收集为一个名为 kwargs 的标准字典。

函数调用中的 **dict（参数解包）：实参前面的 ** 会将字典拆开成 key=value 的形式传入函数

必须遵守的规则

- 所有键必须是字符串: 关键字参数本来就是用名字传递的，非字符串键无法映射到参数名
- 键在所有参数中必须唯一（不能重复）: 关键字参数不能重复——如果两个来源都提供`x` ，解释器无法决定用哪个


## **可直接用在dict字面量中，可多次使用

In [10]:
{"a": 0, **{"x": 1}, "y": 2, **{"z": 3, "x": 4}}

{'a': 0, 'x': 4, 'y': 2, 'z': 3}

**字面量场景下 允许键重复 ，处理方式与 dict 赋值一致—— 按出现顺序从左到右依次合并，后面的键覆盖前面的键**


### 用 `|` 与 `|=` 合并映射

从Python 3.9开始，映射支持并集运算符

In [11]:
d1 = {"a": 1, "b": 3}
d2 = {"a": 2, "b": 4, "c": 6}
d1 | d2  # 创建新映射

{'a': 2, 'b': 4, 'c': 6}

In [12]:
d1

{'a': 1, 'b': 3}

In [13]:
# 就地更新左操作数
d1 |= d2
d1

{'a': 2, 'b': 4, 'c': 6}

> **`|`** **创建新映射，`|=`** **就地更新；新映射的类型通常与左操作数相同（涉及用户自定义类型时也可能与右操作数相同，这涉及运算符重载规则）。**

## 用模式匹配处理映射

`match/case` 的匹配对象可以是映射。**映射模式看起来像 dict 字面量，但它能匹配** **`collections.abc.Mapping`** **的任何实际子类或虚拟子类实例**。不同类型的模式还可以组合、嵌套——借助析构，模式匹配成为处理嵌套映射与序列等**结构化记录**（如来自 JSON API、MongoDB/PostgreSQL 的半结构化数据）的利器。


In [14]:
def get_creators(record:dict) -> list:
    match record:
        case {'type':'book','api':2,'authors':[*names]}:
            return names
        case {'type':'book','api':1,'author':name}:
            return [name]
        case {'type':'book'}:
            raise ValueError('No authors found')
        case {'type':'movie','director':name}:
            return [name]
        case _:
            raise ValueError('Unknown record type')

b1=dict(api=1,author='J.K Rowling',type='book',title='Harry Potter and the Philosopher''s Stone')
b1

{'api': 1,
 'author': 'J.K Rowling',
 'type': 'book',
 'title': 'Harry Potter and the Philosophers Stone'}

In [15]:
get_creators(b1)

['J.K Rowling']

In [16]:
from collections import OrderedDict
b2=OrderedDict(api=2,type='book',title='Python in a Nutshell',authors='Mark Lutz Herbert S'.split())
get_creators(b2)

['Mark', 'Lutz', 'Herbert', 'S']

**映射模式的关键规则：**

1. **只需部分匹配**即视作成功——这是与序列模式最大的不同（序列模式是全匹配）；
2. 模式中**键的顺序无关紧要**——即使 `b2` 是 `OrderedDict` 也能匹配；
3. 想把多出的 Key:Value 对捕获到一个 dict，用 `**变量`，且**必须放在模式最后**；`**_` 这种画蛇添足的写法无效

> **映射模式 = 部分匹配 + 键序无关 +** **`**rest`** **收尾；内部用** **`get(key, sentinel)`** **检索，所以永远不会「误触发」 defaultdict 的默认值逻辑**



## 映射类型的标准API

`collections.abc` 模块提供了描述 `dict` 及类似类型接口的两个抽象基类（ABCs）：`Mapping` 与 `MutableMapping`。ABC 的主要价值在于：**对标准接口进行文档化和正式化，并为** **`isinstance`** **提供广义上的测试标准**：


In [21]:
from collections import abc

my_dict={}
print(isinstance(my_dict,abc.Mapping))
print(isinstance(my_dict,abc.MutableMapping))

True
True


### 什么是可哈希

> 如果一个对象在其生命周期内有一个**永不改变的哈希码**（依托 `__hash__()` 方法），并且可以与其他对象进行**等值比较**（依托 `__eq__()` 方法），那么这个对象就是可哈希（hashable）的。**相等的可哈希对象，必须拥有相同的哈希码**

各类型可哈希性：

| 类型 | 可哈希？ | 说明 |
|---|:-:|---|
| int / float / bool 等数值类型 | ✅ | |
| str / bytes（不可变扁平类型） | ✅ | |
| tuple | ⚠️ 视内容而定 | 仅当**所有项都可哈希**时 tuple 才可哈希 |
| frozenset | ✅ 总是可哈希 | 定义上就要求每个元素可哈希，所以自身必然可哈希 |
| list / set / dict | ❌ | `hash((1, 2, [30, 40]))` → `TypeError: unhashable type: 'list'` |
| 用户自定义类型实例 | ✅（默认） | 哈希码取自 `id()`，继承的 `__eq__` 只比较对象 ID |


In [22]:
t1 = (1, 2, (30, 40))
hash(t1)

-3907003130834322577

In [24]:
# t2=(1,2,[30,40])
# hash(t2)

# ---------------------------------------------------------------------------
# TypeError                                 Traceback (most recent call last)
# Cell In[23], line 2
#       1 t2=(1,2,[30,40])
# ----> 2 hash(t2)

# TypeError: unhashable type: 'list'


In [25]:
t3=(1,2,frozenset([30,40]))
hash(t3)

5149391500123939311

> **「可哈希」三要素 = 哈希码恒定 + 可等值比较 + 相等则哈希码相同。这也解释了为什么 dict 键必须可哈希——哈希表要靠哈希码定位、靠 `__eq__` 确认。**

### 常用映射方法

在 Python 中，**映射（Mapping）** 的核心实现是字典（`dict`）。字典本质上是“键-值对”（Key-Value）的哈希表结构，它的键（Key）和**集合（Set）**的元素一样，都必须是**不可变且可哈希的（Hashable）**。

下面为你整理了 Python 字典最常见的内置方法与操作表格：

---

### Python 常见映射（字典）方法一览表

| 分类 | 方法 / 语法 | 功能描述 | 示例代码 | 返回值 / 关键注意点 |
| --- | --- | --- | --- | --- |
| **查询** | `d[key]` | 按键获取值 | `d['a']` | 返回对应值；**若键不存在抛出 `KeyError**` |
| **查询** | `d.get(key, default=None)` | 安全获取值 | `d.get('b', 0)` | 键不存在时返回 `default`，**不会报错** |
| **视图** | `d.keys()` | 获取所有键的视图 | `d.keys()` | 返回类似集合的视图对象（支持集合运算） |
| **视图** | `d.values()` | 获取所有值的视图 | `d.values()` | 返回动态视图，原字典修改时自动同步 |
| **视图** | `d.items()` | 获取所有 `(key, value)` 元组 | `for k, v in d.items():` | 常用于 `for` 循环遍历键值对 |
| **增 / 改** | `d[key] = value` | 设置/更新键值对 | `d['a'] = 10` | 键存在则覆盖旧值，不存在则新增 |
| **增 / 改** | `d.update(other)` | 批量更新/合并字典 | `d.update({'b': 2, 'c': 3})` | 原地修改字典，也可写作 `d |= other` (3.9+) |
| **增 / 查** | `d.setdefault(key, default)` | 获取值，不存在则先插入 | `d.setdefault('count', 0)` | 键存在返回已有值；不存在则把 `key: default` 存入并返回 |
| **删除** | `d.pop(key, default)` | 弹出并返回指定键的值 | `val = d.pop('a', None)` | 键不存在且未指定 `default` 时抛出 `KeyError` |
| **删除** | `d.popitem()` | 弹出并返回末尾键值对 | `k, v = d.popitem()` | 返回 `(key, value)`；字典为空时抛出 `KeyError` |
| **删除** | `del d[key]` | 删除指定键（语句） | `del d['a']` | 纯删除无返回值；不存在抛出 `KeyError` |
| **删除** | `d.clear()` | 清空字典内所有元素 | `d.clear()` | 清空后变为 `{}`，保留原字典引用 |
| **构造** | `dict.fromkeys(seq, val)` | 以序列为键快速建表 | `dict.fromkeys(['x', 'y'], 0)` | 所有键指向**同一个**默认值对象（注意可变对象陷阱） |
| **复制** | `d.copy()` | 浅拷贝字典 | `new_d = d.copy()` | 只拷贝第一层结构，嵌套对象仍共享引用 |

---

### 学习时最容易混淆的几个重点

1. **`d[key]` 与 `d.get(key)` 的选择**
* 如果键不存在是**代码逻辑异常**，用 `d[key]`（及时抛出 `KeyError` 暴露 Bug）。
* 如果键不存在是**正常业务分支**，用 `d.get(key, 默认值)` 优雅处理兜底逻辑。


2. **`setdefault` 的典型场景**
* 适合做分组统计（如按分类归集列表）：
```python
grades = {}
grades.setdefault('Math', []).append(95)
grades.setdefault('Math', []).append(88)
# grades 为 {'Math': [95, 88]}

```

## 插入或更新可变的值: setdefault

按 Python 的 Fail-Fast 原则，`d[k]` 访问不存在的键会立即报错。获取默认值可用 `d.get(k, default)`；但**想获取一个可变值并就地更新**时，`setdefault` 才是更优解。

In [26]:
import re

# 1. 准备测试文本（Python 之禅节选）
text = """Beautiful is better than ugly.
Explicit is better than implicit.
Simple is better than complex."""

# 2. 匹配单词字符的正则表达式
WORD_RE = re.compile(r"\w+")

index = {}

#  3. 按行遍历文本,获取(行号,列号)并归集
for line_no, line in enumerate(text.splitlines(), start=1):
    for match in WORD_RE.finditer(line):
        word = match.group()
        column_no = match.start() + 1
        location = (line_no, column_no)

        # 核心用法:如果word不存在,存入空列表并返回,若已存在,则直接返回
        index.setdefault(word, []).append(location)

for word in sorted(index, key=str.upper):
    print(f"{word:<12}  -> {index[word]}")

Beautiful     -> [(1, 1)]
better        -> [(1, 14), (2, 13), (3, 11)]
complex       -> [(3, 23)]
Explicit      -> [(2, 1)]
implicit      -> [(2, 25)]
is            -> [(1, 11), (2, 10), (3, 8)]
Simple        -> [(3, 1)]
than          -> [(1, 21), (2, 20), (3, 18)]
ugly          -> [(1, 26)]


> **「取出可变值再更新」优先用 `setdefault(k, []).append(...)`，查找一次到位；它处理的正是 Fail-Fast 的 `d[k]` 与只读的 `d.get()` 之间的空档。**